In [2]:
# electricity
from variables_to_specify_electricity import *
from training_utilities_2nd_part import *
from training_utilities import *
df, columns_to_normalize, elect_target_col, forecast_avg_target_col_name, avg_target_col_name, No_of_datapoints_in_one_day, start_date, end_date, delta, one_month_days, out_columns, elect_drop_columnss, elect_windows, index_of_one_month, one_month_window_size, date_col_name = variables_to_specify_electricity()

df = df.dropna().reset_index(drop=True)
df = df.drop(range(1344)).reset_index(drop=True)
electricity_df = df
elect_time_steps = 1
convert_time(electricity_df, date_col_name)
electricity_df

,date,day,period,nswprice,nswdemand,vicprice,vicdemand,transfer,class,year,month,hour,minute
0,1970-01-01,1,0.000000,0.085565,0.541803,0.003467,0.422915,0.414912,1,1970,1,0,0
1,1970-01-01,1,0.021277,0.085565,0.506992,0.003467,0.422915,0.414912,1,1970,1,0,0
2,1970-01-01,1,0.042553,0.078209,0.477834,0.003467,0.422915,0.414912,1,1970,1,0,0
3,1970-01-01,1,0.063830,0.064519,0.415204,0.003467,0.422915,0.414912,0,1970,1,0,0
4,1970-01-01,1,0.085106,0.064519,0.351532,0.003467,0.422915,0.414912,0,1970,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
42763,1970-01-01,1,0.914894,0.044224,0.340672,0.003033,0.255049,0.405263,0,1970,1,0,0
42764,1970-01-01,1,0.936170,0.044884,0.355549,0.003072,0.241326,0.420614,0,1970,1,0,0
42765,1970-01-01,1,0.957447,0.043593,0.340970,0.002983,0.247799,0.362281,0,1970,1,0,0
42766,1970-01-01,1,0.978723,0.066651,0.329366,0.004630,0.345417,0.206579,1,1970,1,0,0


# stationary

In [3]:
elect_len_of_training_data_of_stationary_model = 14*No_of_datapoints_in_one_day

train = df[0:elect_len_of_training_data_of_stationary_model] 
test = df[elect_len_of_training_data_of_stationary_model:]

eval_df_first_month, stationary_model1 = stationary_model_with_hptuning(train, test, one_month_window_size, 2, out_columns, elect_target_col, elect_drop_columnss)

sum_training_time_stat1 = eval_df_first_month['training_time'].sum()
print('sum_training_time is: ', sum_training_time_stat1)
print(eval_df_first_month['Testing Error'].mean())
print(eval_df_first_month['mae'].mean())

Model Type: XGBRegressor
Storage Required: 0.13 MB
model storage is : 0.13197708129882812


total_time is:  0.691891333
sum_training_time is:  15.56769148199999
0.0017085930236612086
0.025317110454610624


 # Model reuse

In [16]:
daily_df_avg = get_elect_daily_avg(electricity_df, No_of_datapoints_in_one_day, elect_target_col, avg_target_col_name)

seasonality_periods_acf_ls, seasonality_periods_acf, segmented_daily_df_avg, filtered_most_similar_dict_wass, filtered_most_similar_dict_tvd, forecast_daily_df_avg, segmented_forecast_daily_df_avg, filtered_forecasted_most_similar_dict_wass, filtered_forecasted_most_similar_dict_tvd = get_seasonality_segments_and_similarities(daily_df_avg, avg_target_col_name, forecast_avg_target_col_name, 14)

Detected seasonality periods (ACF): [  4   7  10  14  17  21  23  27  34  42  49  53  56  58  63  68  72  76
  80  85  92  97 104 107 112 114 121]
median_value is:  58


In [17]:
df_copy = electricity_df[[elect_target_col]]
target_col = elect_target_col
time_steps = elect_time_steps

df_copy['date'] = pd.to_datetime(df_copy.index)
multiplier = No_of_datapoints_in_one_day
x = 14* multiplier
window_len_=[x]

drift_results_df_ls = []
for i in window_len_:
    start_drift_detection_time = timeit.default_timer()
    drift_results_df = detect_drift_univariate(
        df_copy,
        target_col=elect_target_col,
        window_lengths=window_len_,
        arima_order=(1, 0, 0)
    )
    drift_results_df_ls.append(drift_results_df)
    drift_detection_time = timeit.default_timer() - start_drift_detection_time
    num_true = drift_results_df['drift_detected'].sum()
    print("i is: ", i, " and the Number of True values in 'drift_detected':", num_true, " total number of rows are : ", len(drift_results_df))
    print("drift detection time is: ", drift_detection_time)
    drift_results_df = drift_results_df_ls[0]
    drift_indices = list(drift_results_df.index[drift_results_df['drift_detected']])
    print("indices are: ", drift_indices)

Fold 0: Train size=134, Test size=134
Fold 1: Train size=268, Test size=134
Fold 2: Train size=402, Test size=134
Fold 3: Train size=536, Test size=134
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=134, Test size=134
Fold 1: Train size=268, Test size=134
Fold 2: Train size=402, Test size=134
Fold 3: Train size=536, Test size=134
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=134, Test size=134
Fold 1: Train size=268, Test size=134
Fold 2: Train size=402, Test size=134
Fold 3: Train size=536, Test size=134
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=134, Test size=134
Fold 1: Train size=268, Test size=134
Fold 2: Train size=402, Test size=134
Fold 3: Train size=536, Test size=134
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=134, Test size=134
Fold 1: Train size=268, Test size=134
Fold 2: Train size=402, Test size=134
Fold 3: Train size=536, Test size=134
Skipping fold 4: Insufficien

In [18]:
eval_df_monthly2, avg_ml_storage1 = new_copied_reuse_with_hptuning_no_while_loop_with_drift(filtered_most_similar_dict_wass, stationary_model1, elect_len_of_training_data_of_stationary_model,electricity_df, "SA", elect_target_col, elect_drop_columnss, elect_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices, 2)


window is:  672
i/window is :  1.0
Model Type: XGBRegressor
Storage Required: 0.13 MB


window is:  1344
i/window is :  2.0
Model Type: XGBRegressor
Storage Required: 0.13 MB


window is:  2016
i/window is :  3.0
similar_month_index is :  1
month_index:  3




window is:  2688
i/window is :  4.0
Model Type: XGBRegressor
Storage Required: 0.12 MB


window is:  3360
i/window is :  5.0
similar_month_index is :  3
previous_model_i is :  3360
math.floor(previous_model_i/window) is:  5
len(models_ls) is: 4
Model Type: XGBRegressor
Storage Required: 0.10 MB


window is:  4032
i/window is :  6.0
Model Type: XGBRegressor
Storage Required: 0.10 MB


window is:  4704
i/window is :  7.0
similar_month_index is :  5
previous_model_i is :  4704
math.floor(previous_model_i/window) is:  7
len(models_ls) is: 6
Model Type: XGBRegressor
Storage Required: 0.09 MB


window is:  5376
i/window is :  8.0
similar_month_index is :  0
month_index:  8




window is:  6048
i/window is :  9.0
similar_month_index is 

In [19]:
eval_df_monthly2, avg_ml_storage2 = new_copied_reuse_with_hptuning_no_while_loop_with_drift(filtered_most_similar_dict_tvd, stationary_model1, elect_len_of_training_data_of_stationary_model,electricity_df, "SA", elect_target_col, elect_drop_columnss, elect_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices, 2)

window is:  672
i/window is :  1.0
Model Type: XGBRegressor
Storage Required: 0.13 MB


window is:  1344
i/window is :  2.0
similar_month_index is :  0
month_index:  2




window is:  2016
i/window is :  3.0
Model Type: XGBRegressor
Storage Required: 0.11 MB


window is:  2688
i/window is :  4.0
Model Type: XGBRegressor
Storage Required: 0.12 MB


window is:  3360
i/window is :  5.0
similar_month_index is :  0
month_index:  5




window is:  4032
i/window is :  6.0
Model Type: XGBRegressor
Storage Required: 0.10 MB


window is:  4704
i/window is :  7.0
Model Type: XGBRegressor
Storage Required: 0.09 MB


window is:  5376
i/window is :  8.0
Model Type: XGBRegressor
Storage Required: 0.09 MB


window is:  6048
i/window is :  9.0
similar_month_index is :  2
previous_model_i is :  6048
math.floor(previous_model_i/window) is:  9
len(models_ls) is: 8
Model Type: XGBRegressor
Storage Required: 0.11 MB


window is:  6720
i/window is :  10.0
similar_month_index is :  6
month_index:  10




wind

In [20]:
eval_df_monthly2, avg_ml_storage3 = new_copied_reuse_with_hptuning_no_while_loop_with_drift(filtered_forecasted_most_similar_dict_wass, stationary_model1, elect_len_of_training_data_of_stationary_model,electricity_df, "ES", elect_target_col, elect_drop_columnss, elect_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices, 2)

window is:  672
Model Type: XGBRegressor
Storage Required: 0.13 MB


window is:  1344
Model Type: XGBRegressor
Storage Required: 0.13 MB


window is:  2016
Model Type: XGBRegressor
Storage Required: 0.11 MB


window is:  2688
similar_month_index is :  1
month_index:  3




window is:  3360
similar_month_index is :  1
month_index:  4




window is:  4032
Model Type: XGBRegressor
Storage Required: 0.10 MB


window is:  4704
similar_month_index is :  2
month_index:  6




window is:  5376
Model Type: XGBRegressor
Storage Required: 0.09 MB


window is:  6048
similar_month_index is :  4
previous_model_i is :  6048
math.floor(previous_model_i/window) is:  9
len(models_ls) is: 8
Model Type: XGBRegressor
Storage Required: 0.11 MB


window is:  6720
similar_month_index is :  1
month_index:  9




window is:  7392
similar_month_index is :  3
previous_model_i is :  7392
math.floor(previous_model_i/window) is:  11
len(models_ls) is: 10
Model Type: XGBRegressor
Storage Required: 0.10 MB


window is

In [21]:
eval_df_monthly2, avg_ml_storage4 = new_copied_reuse_with_hptuning_no_while_loop_with_drift(filtered_forecasted_most_similar_dict_tvd, stationary_model1, elect_len_of_training_data_of_stationary_model,electricity_df, "ES", elect_target_col, elect_drop_columnss, elect_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices, 2)

window is:  672
Model Type: XGBRegressor
Storage Required: 0.13 MB


window is:  1344
Model Type: XGBRegressor
Storage Required: 0.13 MB


window is:  2016
Model Type: XGBRegressor
Storage Required: 0.11 MB


window is:  2688
similar_month_index is :  1
month_index:  3




window is:  3360
similar_month_index is :  1
month_index:  4




window is:  4032
Model Type: XGBRegressor
Storage Required: 0.10 MB


window is:  4704
Model Type: XGBRegressor
Storage Required: 0.09 MB


window is:  5376
similar_month_index is :  4
previous_model_i is :  5376
math.floor(previous_model_i/window) is:  8
len(models_ls) is: 7
Model Type: XGBRegressor
Storage Required: 0.11 MB


window is:  6048
similar_month_index is :  1
month_index:  8




window is:  6720
similar_month_index is :  1
month_index:  9




window is:  7392
similar_month_index is :  1
month_index:  10




window is:  8064
Model Type: XGBRegressor
Storage Required: 0.10 MB


window is:  8736
Model Type: XGBRegressor
Storage Required: 0.09 

In [22]:
avg_ml_storage_reuse = (avg_ml_storage1+avg_ml_storage2+avg_ml_storage3+avg_ml_storage4)/4
print(avg_ml_storage_reuse)

0.0712915854983001


# informed

In [23]:
informed_update(stationary_model1,electricity_df, target_col, elect_drop_columnss,time_steps, seasonality_periods_acf,No_of_datapoints_in_one_day, drift_indices, 2)

window is:  672
Model Type: XGBRegressor
Storage Required: 0.13 MB
window is:  1344
window is:  2016
Model Type: XGBRegressor
Storage Required: 0.11 MB
window is:  2688
window is:  3360
window is:  4032
Model Type: XGBRegressor
Storage Required: 0.10 MB
window is:  4704
window is:  5376
window is:  6048
Model Type: XGBRegressor
Storage Required: 0.11 MB
window is:  6720
window is:  7392
Model Type: XGBRegressor
Storage Required: 0.10 MB
window is:  8064
window is:  8736
window is:  9408
window is:  10080
window is:  10752
Model Type: XGBRegressor
Storage Required: 0.08 MB
window is:  11424
Model Type: XGBRegressor
Storage Required: 0.08 MB
window is:  12096
window is:  12768
Model Type: XGBRegressor
Storage Required: 0.09 MB
window is:  13440
Model Type: XGBRegressor
Storage Required: 0.06 MB
window is:  14112
Model Type: XGBRegressor
Storage Required: 0.08 MB
window is:  14784
Model Type: XGBRegressor
Storage Required: 0.07 MB
window is:  15456
window is:  16128
window is:  16800
wind

# periodical

In [24]:
periodical_retraining_with_hptuning(2, electricity_df, elect_windows, out_columns, elect_target_col, elect_drop_columnss)

window is : 240
window size is :  240
Model Type: XGBRegressor
Storage Required: 0.09 MB
Model Type: XGBRegressor
Storage Required: 0.07 MB
Model Type: XGBRegressor
Storage Required: 0.10 MB
Model Type: XGBRegressor
Storage Required: 0.10 MB
Model Type: XGBRegressor
Storage Required: 0.05 MB
Model Type: XGBRegressor
Storage Required: 0.07 MB
Model Type: XGBRegressor
Storage Required: 0.06 MB
Model Type: XGBRegressor
Storage Required: 0.05 MB
Model Type: XGBRegressor
Storage Required: 0.08 MB
Model Type: XGBRegressor
Storage Required: 0.07 MB
Model Type: XGBRegressor
Storage Required: 0.08 MB
Model Type: XGBRegressor
Storage Required: 0.07 MB
Model Type: XGBRegressor
Storage Required: 0.06 MB
Model Type: XGBRegressor
Storage Required: 0.07 MB
Model Type: XGBRegressor
Storage Required: 0.08 MB
Model Type: XGBRegressor
Storage Required: 0.06 MB
Model Type: XGBRegressor
Storage Required: 0.06 MB
Model Type: XGBRegressor
Storage Required: 0.06 MB
Model Type: XGBRegressor
Storage Required: 0

([          Training dataset     Testing dataset       mae       mse      rmse  \
  0    trained on window i-1  tested on window i  0.027567  0.000999  0.031601   
  1    trained on window i-1  tested on window i  0.030107  0.001277  0.035733   
  2    trained on window i-1  tested on window i  0.025077  0.001016  0.031876   
  3    trained on window i-1  tested on window i  0.044878  0.002454  0.049538   
  4    trained on window i-1  tested on window i  0.015668  0.000567  0.023808   
  ..                     ...                 ...       ...       ...       ...   
  172  trained on window i-1  tested on window i  0.004508  0.000086  0.009296   
  173  trained on window i-1  tested on window i  0.001012  0.000003  0.001843   
  174  trained on window i-1  tested on window i  0.002756  0.000051  0.007159   
  175  trained on window i-1  tested on window i  0.001638  0.000018  0.004223   
  176  trained on window i-1  tested on window i  0.001838  0.000007  0.002665   
  
             